# **Clustering**
## Insurance company customers' segmentation 

## Dataset description

- **CustID**: numeric - customer ID
- **FirstPolYear**: numeric - year of the customer's first policy. Maybe considered the first year as a customer
- **BirthYear**: numeric - birth year of the customer. The current year of the database is 2016
- **EducDeg**: categorical - academic degree
- **MonthSal**: numerical - monthly gross salary (€)
- **GeoLivArea**: numerical - codes about the area of living. No additional information is available for these codes
- **Children**: numerical - indication if the customer has children (0: no, 1: yes)
- **CustMonVal**: numerical - customer monetary value (CMV). CMV = (annual profit from the customer) x (number of years since a customer) - (acquisition cost)
- **ClaimsRate**: numerical - claims rate. Amount paid by the insurance company (€)/Premiums(€) (in the last two years)
- **PremMotor**: numerical - premiums in the Line of Business (LOB) Motor (€)
- **PremHousehold**: numerical - premiums in the LOB Household (€)
- **PremHealth**: numerical - premiums in the LOB Health (€)
- **PremLife**: numerical - premiums in the LOB Life (€)
- **PremWork**: numerical - premiums in the LOB Work (€)

<br>NOTES about all Premiums:
- Annual premiums (2016)
- Negative premiums may manifest reversals occurred in the current year, paid in previous one(s)

___

## **Initializations and data loading:**

In [ ]:
# Uncomment to install the packages.

#!pip install category_encoders
#!pip install git+https://github.com/sevamoo/SOMPY.git
#!pip install yellowbrick
#!pip install graphviz

In [ ]:
# Packages

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker
import seaborn as sns
import category_encoders as ce
import collections
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn import preprocessing
from sklearn.decomposition import PCA, KernelPCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer, InterclusterDistance
from sklearn.manifold import TSNE
import time
import hashlib

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.cluster.vq import vq, kmeans, whiten

from sklearn import metrics
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import sompy
from sompy.visualization.mapview import View2D
from sompy.visualization.bmuhits import BmuHitsView
from sompy.visualization.hitmap import HitMapView

import matplotlib.cm as cm
from sklearn.metrics import silhouette_score, silhouette_samples, pairwise_distances

from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, DBSCAN, estimate_bandwidth, KMeans, AgglomerativeClustering

from collections import Counter
from os.path import join

from sklearn.base import clone 
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import train_test_split
import graphviz
import logging
sns.set()

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import time
t1 = time.perf_counter()

In [ ]:
%matplotlib inline
np.set_printoptions(precision=5, suppress=True)  # suppress scientific float notation
logging.getLogger('matplotlib.font_manager').disabled = True

In [ ]:
def get_ss(df):
    """Computes the sum of squares for all variables given a dataset
    """
    ss = np.sum(df.var()*(df.count() - 1))
    return ss  # return sum of sum of squares of each df variable

In [ ]:
def get_r2(df, labels):
    """R² of a cluster solution, over the clustering variables only.

    labels is a column name of df or an array of labels. Label columns, BMU
    indices and one-hot dummies are excluded, so they cannot inflate R².
    """
    groups = df[labels] if isinstance(labels, str) else pd.Series(labels, index=df.index)
    features = df[[c for c in df.columns if c in num_vars]]
    sst = get_ss(features)  # total sum of squares
    ssw_labels = features.groupby(groups).apply(get_ss)  # within-cluster SS per label
    ssb = sst - np.sum(ssw_labels)  # SST = SSW + SSB
    return ssb / sst

In [ ]:
def get_ss_variables(df):
    """Get the SS for each variable
    """
    ss_vars = df.var()*(df.count() - 1)
    return ss_vars

In [ ]:
def get_r2_variables(df, labels):
    """Get the R² for each variable
    """
    sst_vars = get_ss_variables(df)
    ssw_vars = np.sum(df.groupby(labels).apply(get_ss_variables))
    return 1 - ssw_vars/sst_vars

In [ ]:
# Adapted from:
# https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.htmlsphx-glr-auto-examples-cluster-plot-kmeans-silhouette-analysis-py


def get_avg_silhouette(df, model, range_clusters=range(1, 11)):
    # Storing average silhouette metric
    avg_silhouette = []
    for nclus in range_clusters:
        # Skip nclus == 1
        if nclus == 1:
            continue
    
        # Create a figure
        fig = plt.figure(figsize=(13, 7))
        # Initialize the model object with n_clusters value and a random generator
        # setting the seed for reproducibility.
        modelclust = clone(model).set_params(n_clusters=nclus) 
        #modelclust = model.set_params(nclus)
        cluster_labels = modelclust.fit_predict(df)
        # why we only use numerical variables.

        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed clusters
        silhouette_avg = silhouette_score(df, cluster_labels)
        avg_silhouette.append(silhouette_avg)
        print(f"For n_clusters = {nclus}, the average silhouette_score is : {silhouette_avg}")

        # Compute the silhouette scores for each sample
        sample_silhouette_values = silhouette_samples(df, cluster_labels)

        y_lower = 10
        for i in range(nclus):
            # Aggregate the silhouette scores for samples belonging to cluster i, and sort them
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
            ith_cluster_silhouette_values.sort()
        
            # Get y_upper to demarcate silhouette y range size
            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i
        
            # Filling the silhouette
            color = cm.nipy_spectral(float(i) / nclus)
            plt.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)

            # Label the silhouette plots with their cluster numbers at the middle
            plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            # Compute the new y_lower for next plot
            y_lower = y_upper + 10  # 10 for the 0 samples

        plt.title("The silhouette plot for the various clusters.")
        plt.xlabel("The silhouette coefficient values")
        plt.ylabel("Cluster label")

        # The vertical line for average silhouette score of all the values
        plt.axvline(x=silhouette_avg, color="red", linestyle="--")
    
        # The silhouette coefficient can range from -1, 1
        xmin, xmax = np.round(sample_silhouette_values.min() -0.1, 2), np.round(sample_silhouette_values.max() + 0.1, 2)
        plt.xlim([xmin, xmax])
    
        # The (nclus+1)*10 is for inserting blank space between silhouette
        # plots of individual clusters, to demarcate them clearly.
        plt.ylim([0, len(X[num_vars]) + (nclus + 1) * 10])

        plt.yticks([])  # Clear the yaxis labels / ticks
        plt.xticks(np.arange(xmin, xmax, 0.1))
    
    return avg_silhouette

In [ ]:
# Count observations per cluster

def get_cluster_analysis(df, label, allDistances):
    """
    df: the dataframe.
    label: the label of our target
    allDistances: distances to clusters.
    """
    # We need to define the centroids of our clusters.
    centroids = df.groupby(label, as_index=False).mean()

    # frequencies:
    freqByCluster = df.groupby(label).size()

    # Compute Magnitude
    df['distanceToCentroid'] = np.min(allDistances,axis=1)
    magnitude = df.groupby(label).sum()["distanceToCentroid"] 
    df.drop(columns=['distanceToCentroid'], inplace=True)

    # figure and axes
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

    # draw plots
    pd.plotting.parallel_coordinates(centroids, class_column = label, colormap="viridis", 
                                     axvlines=True, lw=2, ax=ax1)
    sns.countplot(x=df[label], color='grey', ax=ax2)
    sns.barplot(x=magnitude.index, y=magnitude.values, color='grey', ax=ax3)
    sns.regplot(x=freqByCluster, y=magnitude, scatter=True, seed=random_state, truncate=False, ci=None, ax=ax4)

    # customizations
    ax1.tick_params(labelrotation=45)

    # ax2 Decoration
    fmt = "{x:,.0f}"
    tick = ticker.StrMethodFormatter(fmt)
    ax2.yaxis.set_major_formatter(tick)
    for index, data in enumerate(freqByCluster):
        ax2.text(x=index-0.2 , y=data+50 , s=f"{data}" , fontdict=dict(fontsize=plots_barTexts_fontSize))
    ax2.set_title("Cluster cardinality", fontsize=plots_Title_fontSize)
    ax2.set_xlabel("Cluster")
    ax2.set_ylabel("Frequency in cluster")

    # ax3 Decoration
    ax3.yaxis.set_major_formatter(tick)
    for index, data in enumerate(magnitude):
        ax3.text(x=index-0.2 , y=data+50 , s=f"{data:,.0f}" , fontdict=dict(fontsize=plots_barTexts_fontSize))
    ax3.set_title("Cluster magnitude", fontsize=plots_Title_fontSize)
    ax3.set_xlabel("Cluster")
    ax3.set_ylabel("Sum of distances to centroid")

    # ax4 Decoration
    ax4.xaxis.set_major_formatter(tick)
    ax4.yaxis.set_major_formatter(tick)
    sns.despine()
    ax4.set_title("Cardinality vs Magnitude", fontsize=plots_Title_fontSize)
    ax4.set_xlabel("Cardinality")
    ax4.set_ylabel("Magnitude")
    fig.tight_layout()
    plt.show();

In [ ]:
# Function to find outliers on columns based on percentile
def removeOutliers(df, colList, lowPercentile=0.05, highPercentile=0.95, verbose=False):
    quant_df = df[colList].quantile([lowPercentile, highPercentile])
    if verbose:
        print(quant_df)
    for name in list(df[colList].columns):
        df_out = df[~((df[name] >= quant_df.loc[lowPercentile, name]) & (df[name] <= quant_df.loc[highPercentile, name]))]
        df = df[(df[name] >= quant_df.loc[lowPercentile, name]) & (df[name] <= quant_df.loc[highPercentile, name])]
    return df, df_out

In [ ]:
random_state=2022

subPlots_Title_fontSize = 12
subPlots_xAxis_fontSize = 10
subPlots_yAxis_fontSize = 10
subPlots_label_fontSize = 10
heatmaps_text_fontSize = 8

plots_Title_fontSize = 14
plots_Title_textColour = 'black'

plots_Legend_fontSize = 12
plots_Legend_textColour = 'black'

plots_barTexts_fontSize = 8

In [ ]:
# Loading the dataset

ds = pd.read_sas('data/a2z_insurance.sas7bdat', format='sas7bdat')

In [ ]:
ds.info()

In [ ]:
#  Visualizing summary statistics

ds.describe(include='all').T

In [ ]:
# Show top rows
ds.head()

In [ ]:
ds.shape

___

# **1. Data Exploration:**

__The problems we might face includes:__
- Duplicates.
- Missing values.
- Strange values.

In [ ]:
# replace "" by nans
ds.replace("", np.nan, inplace=True)

In [ ]:
ds.isna().sum()

In [ ]:
ds.duplicated().sum()

We can see that our dataset does not contain any duplicates.

In [ ]:
ds.drop(columns=['CustID'], inplace=True)

Dropping the 'ID' column as it does not provide any informations.

In [ ]:
ds.head(1)

### **The variable 'FirstPolYear':**

In [ ]:
ds['FirstPolYear'].unique()

We need to drop the observations with values equal to 53784. 

In [ ]:
ds = ds[~ds['FirstPolYear'].isin([53784.0, np.nan, 'nan', 'Nan'])]

In [ ]:
ds['FirstPolYear'].unique()

In [ ]:
ds.shape

### The variable 'BirthYear':

In [ ]:
ds['BirthYear'].unique()

In [ ]:
ds['BirthYear'].value_counts()

We need to drop the birth year equal to 1028, and the nans.

In [ ]:
ds = ds[~ds['BirthYear'].isin([1028.0, np.nan, 'nan', 'Nan'])]

In [ ]:
ds['BirthYear'].unique()

In [ ]:
ds.shape

### **The variable  'EducDeg':**

In [ ]:
ds['EducDeg'].unique()

In [ ]:
ds['EducDeg'].value_counts()

Removing the b infront of the category name.

In [ ]:
ds['EducDeg'] = ds['EducDeg'].astype(str)
ds['EducDeg'] = ds['EducDeg'].str.replace(r'b', '')
ds['EducDeg'] = ds['EducDeg'].str.replace(r'[^a-zA-Z ]', '')
ds['EducDeg'] = ds['EducDeg'].str.strip()

In [ ]:
# Counting the genders in the dataset.

g=sns.countplot(data=ds, x="EducDeg", color="cornflowerblue")
plt.show();

We can see that most of the people in our dataset have either a Bsc or a Msc degree.

We need to get rid of the nans.

In [ ]:
ds = ds[~ds['EducDeg'].isin([np.nan, 'nan'])]

In [ ]:
ds['EducDeg'].unique()

In [ ]:
ds.shape

### **The variable 'MonthSal':**

In [ ]:
ds['MonthSal'].value_counts()

In [ ]:
ds = ds[~ds['MonthSal'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'MonthSal'

g=sns.displot(data=ds, x="MonthSal", kde=True)
g.set(xlim=(None, 10_000))
plt.show();

We set an xlim value for the figure above because it used to shows the presence of outliers values that are very far from the true distribution of salaries.

In [ ]:
ds.shape

### **The variable 'GeoLivArea':**

In [ ]:
ds['GeoLivArea'].unique()

In [ ]:
ds['GeoLivArea'].value_counts()

We need to remove the nans values if they exist.

In [ ]:
ds = ds[~ds['GeoLivArea'].isin([np.nan, 'nan'])]

In [ ]:
# Counting the 'GeoLivArea' in the dataset.

sns.countplot(data=ds, x="GeoLivArea", color="cornflowerblue")
plt.show();

As we can see most of the observations in our dataset live in the area 4.

In [ ]:
ds.shape

### **The variable 'Children':**

In [ ]:
ds['Children'].unique()

In [ ]:
ds['Children'].value_counts()

Again we need to get rid of the nans.

In [ ]:
ds = ds[~ds['Children'].isin([np.nan, 'nan'])]

In [ ]:
# Counting the 'Children' in the dataset.

sns.countplot(data=ds, x="Children", color="cornflowerblue")
plt.show()

We can see that most people in the dataset have children.

In [ ]:
ds['Children'].unique()

In [ ]:
ds.shape

### The variable 'CustMonVal':

In [ ]:
ds['CustMonVal'].isna().sum()

In [ ]:
ds['CustMonVal'].unique()

In [ ]:
ds = ds[~ds['CustMonVal'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'CustMonVal'

g=sns.histplot(data=ds, x="CustMonVal", kde=True)
g.set(xlim=(-2_000, 2_000))
plt.show();

We set an xlim value for the plot above because it used to show observations with very small negative values, they might be outliers we will decide how to deal with them in the next steps.

In [ ]:
ds.shape

### **The variable 'ClaimsRate':**

In [ ]:
ds['ClaimsRate'].isna().sum()

In [ ]:
ds['ClaimsRate'].unique()

This variable does not contains any nans.
We will plot its histogram.

In [ ]:
# Checking the histogram of the variable 'ClaimsRate'

g=sns.displot(data=ds, x="ClaimsRate", kde=False)
g.set(xlim=(0, 2))
plt.show()

We can see that the variable contains some outliers with claims Rate above 250. we set a value of xlim in order to obtain a more intersting plot.

In [ ]:
ds.shape

### **The variable 'PremMotor':**

In [ ]:
ds['PremMotor'].isna().sum()

In [ ]:
ds['PremMotor'].unique()

In [ ]:
ds = ds[~ds['PremMotor'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'PremMotor'

g = sns.displot(data=ds, x="PremMotor", kde=True)
g.set(xlim=(None, 1000))
plt.show()

The variable seems to contains some outliers. the figure above have a value set to xlim.

In [ ]:
ds.shape

### **The variable 'PremHousehold':**

In [ ]:
ds['PremHousehold'].isna().sum()

In [ ]:
ds['PremHousehold'].unique()

In [ ]:
ds = ds[~ds['PremHousehold'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'PremHousehold'

g=sns.displot(data=ds, x="PremHousehold", kde=True)
g.set(xlim=(None, 5_000))
plt.show();

The distribution is right skewed as it contains very big Household premiums values, we set a value for xlim in order to get a more visible histogram.

In [ ]:
ds.shape

### **The variable 'Premhealth':**

In [ ]:
ds['PremHealth'].isna().sum()

In [ ]:
ds['PremHealth'].unique()

In [ ]:
ds['PremHealth'].value_counts()

In [ ]:
ds = ds[~ds['PremHealth'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'PremHealth'

g=sns.displot(data=ds, x="PremHealth", kde=True)
g.set(xlim=(-500, 1_000))
plt.show();

Right skewed distribution with very high values. The figure above does not show this issue as we set a value for xlim.

In [ ]:
ds.shape

### **The variable 'PremLife':**

In [ ]:
ds['PremLife'].isna().sum()

In [ ]:
ds['PremLife'].unique()

In [ ]:
ds = ds[~ds['PremLife'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'PremLife'

sns.displot(data=ds, x="PremLife", kde=True)
plt.show()

Right skewed distribution, but does not seem to contain outliers.

In [ ]:
ds.shape

### **The variable 'PremWork':**

In [ ]:
ds['PremWork'].isna().sum()

In [ ]:
ds['PremWork'].unique()

In [ ]:
ds = ds[~ds['PremWork'].isin([np.nan, 'nan'])]

In [ ]:
# Checking the histogram of the variable 'PremWork'

g = sns.displot(data=ds, x="PremWork", kde=True)
g.set(xlim=(None, 1_000))
plt.show();
## set limit here


Right skewed distribution. We set a value to xlim in order to get a visible histogram.

In [ ]:
ds.shape

### Checking for missing values:

In [ ]:
ds.isna().sum()

We now have no missing data left.

In [ ]:
# HISTOGRAM (ALL NUMERIC VARIABLES)

# Identify all numeric variables in the dataset
numerical=ds.select_dtypes(include=[np.number]).columns.tolist()

# Draw
fig, ax = plt.subplots(2, 5, figsize=(18,6))
for var, subplot in zip(ds[numerical], ax.flatten()):
    g = sns.histplot(data=ds,
                bins=20,
                 x=var,
                 ax=subplot,
                 kde=False)

# Decoration
sns.despine()
plt.rc('axes', labelsize=subPlots_label_fontSize)
fig.suptitle("Histograms with 20 bins of all numeric variables", fontsize=plots_Title_fontSize)
plt.tight_layout()
plt.show();

The presence of outliers make the histograms not visible enough.

___

# **2. Features Engineering:**

### __Converting 'BirthYear' into 'Age':__

We convert the variable 'BirthYear' into 'Age'.

In [ ]:
ds['Age'] = 2016 - ds['BirthYear']

In [ ]:
ds.drop(columns=['BirthYear'], inplace=True)

In [ ]:
# Checking the histogram of the variable 'Age'.

sns.histplot(data=ds, x="Age", bins=8, kde=True)
plt.show()

The distribution of custumers ages is almost uniformly distributed.

### __Converting the variable 'FirstPolYear' into 'YearsAsCustomer':__

We convert the variable 'FirstPolYear' into 'YearsAsCustumer'

In [ ]:
ds['YearsAsCustomer'] = 2016 - ds['FirstPolYear']

In [ ]:
ds.drop(columns=['FirstPolYear'], inplace=True)

In [ ]:
# Checking the histogram of the variable 'YearsAsCustumer'.

sns.histplot(data=ds, x="YearsAsCustomer", bins=10, kde=True)
plt.show()

In [ ]:
ds.shape

___

# **3. Data Prepararion:**

In [ ]:
X = ds.copy(deep=True)

In [ ]:
X.shape

## __Encoding categorical variables:__

Encoding categorical variables to dummy variables.
For more information the different methods to enconde categorical variables
check https://contrib.scikit-learn.org/category_encoders/

In [ ]:
cat_vars = ['EducDeg', 'GeoLivArea', 'Children']
ce_one_hot = ce.OneHotEncoder(cols = cat_vars, use_cat_names=True)
X = ce_one_hot.fit_transform(X)

In [ ]:
X.shape

## __Removing numerical outliers:__

In [ ]:
num_vars = ["MonthSal", "CustMonVal", "ClaimsRate", "PremMotor", "PremHousehold", "PremHealth", 
            "PremLife", "PremWork"]

In [ ]:
X[num_vars].describe().T

In [ ]:
X_out, X_outliers = removeOutliers(X, num_vars, lowPercentile=0.02, highPercentile=0.95, verbose=True)

In [ ]:
print(X_out.shape)
print(X_outliers.shape)

In [ ]:
X_out[num_vars].describe().T

In [ ]:
X_out[num_vars].hist(color='b', alpha=0.5, bins=10, figsize = (15,20))
plt.show()

In [ ]:
X = X_out.copy(deep=True)

## __Scaling the numerical variables:__

Get a copy of the dataframe before normalization for results' analysis.

In [ ]:
X_beforeNorm = X.copy(deep=True)

In [ ]:
num_vars += ['Age', 'YearsAsCustomer'] 

In [ ]:
# Normalize all columns
# StandardScaler returns an array, so the dataframe must be recreated
X[num_vars] = pd.DataFrame(StandardScaler().fit_transform(X[num_vars].values), 
                           columns=num_vars,index=X.index)

In [ ]:
# Check dataframe structure and statistics after all transformations
print(X.shape)
X[num_vars].describe().T

- The modelling dataset has 20 columns, all numeric.
- The clustering variables are standardised to mean 0 and standard deviation 1.

___

# **4. Analyze Principal Components:**

Visualizing the PCA with only two components.

In [ ]:
pca2 = PCA(n_components = 2, random_state=random_state)
pca2.fit(X[num_vars])
X_pca2 = pca2.transform(X[num_vars])
print(X_pca2.shape)

# Scatter plot of the two Principal Components by the "target" to see if there is any pattern
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=X["MonthSal"], edgecolor='none', alpha=0.5, cmap='viridis')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.colorbar()
plt.show();

- Two principal components, coloured by monthly salary.

In [ ]:
# Fit the PCA algorithm to the dataset. 
pca = PCA().fit(X)

# Output PCA table
pcaDF=pd.DataFrame(
    {"Eigenvalue": pca.explained_variance_,
     "Difference": np.insert(np.diff(pca.explained_variance_), 0, 0),
     "Proportion": pca.explained_variance_ratio_,
     "Cumulative": 100*np.cumsum(pca.explained_variance_ratio_)},
    index=range(1, pca.n_components_ + 1)
)

pcaDF

- The first two components explain only 43.4% of the variance, so a 2-D view cannot settle the number of clusters.

In [ ]:
# figure and axes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# draw plots
ax1.plot(pca.explained_variance_, marker=".", markersize=12)
ax2.plot(pca.explained_variance_ratio_, marker=".", markersize=12, label="Proportion")
ax2.plot(np.cumsum(pca.explained_variance_ratio_), marker=".", markersize=12, linestyle="--", label="Cumulative")

# customizations
ax1.legend()
ax2.legend()
ax1.set_title("Scree Plot", fontsize=14)
ax2.set_title("Variance Explained", fontsize=14)
ax1.set_ylabel("Eigenvalue")
ax2.set_ylabel("Proportion")
ax1.set_xlabel("Components")
ax2.set_xlabel("Components")
ax1.set_xticks(range(0, pca.n_components_, 2))
ax1.set_xticklabels(range(1, pca.n_components_ + 1, 2))
ax2.set_xticks(range(0, pca.n_components_, 2))
ax2.set_xticklabels(range(1, pca.n_components_ + 1, 2))

plt.show();

In [ ]:
np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.9)

- checking the number of components that have an acumulated variance bigger that 90%.

___

# **5. KMeans:**

### *Characteristics:*
- *Number of clusters* need to be set apriori.
- One of the *fastest* clustering algorithms.
- The results *depend on the initialization* (stochastic).
- Prone to *local optima*.
- Favors *convex* (round shape) and *isotropic* (same shape) clusters.

## __Defining the number of clusters.__

In [ ]:
range_clusters = range(1, 11)

In [ ]:
inertia = []
for n_clus in range_clusters:  # iterate over desired ncluster range
    kmclust = KMeans(n_clusters=n_clus, init='k-means++', n_init=20, random_state=random_state)
    kmclust.fit(X[num_vars])
    inertia.append(kmclust.inertia_)  # save the inertia of the given cluster solution

**Inertia (within-cluster sum-of-squares distance) Formula:**
$$\sum_{j=0}^{C}\sum_{i=0}^{n_j}(||x_i - \mu_j||^2)$$
, where:

$C$: Set of identified clusters.

$n_j$: Set of observations belonging to cluster $j$.

$x_i$: Observation $i$.

$\mu_j$: Centroid of cluster $j$.

In [ ]:
# The inertia plot.
plt.figure(figsize=(9,5))
plt.plot(range_clusters, inertia, marker=".", markersize=12)
plt.ylabel("Inertia: SSW")
plt.xlabel("Number of clusters")
plt.title("Inertia plot over clusters", size=15)
plt.show();

- The inertia curve bends most clearly at 4 clusters.

**Silhouette Coefficient formula for a single sample:**
$$s = \frac{b - a}{max(a, b)}$$
, where:
- $a$: The mean distance between a sample and all other points in the same cluster.
- $b$: The mean distance between a sample and all other points in the next nearest cluster

In [ ]:
# Calclating the silhouette score.

avg_silhouette = get_avg_silhouette(X[num_vars], KMeans(init='k-means++', n_init=20, random_state=random_state))

In [ ]:
# The average silhouette plot
plt.figure(figsize=(9,5))
plt.plot(range(2,11), avg_silhouette, marker=".", markersize=12)
plt.ylabel("Average silhouette")
plt.xlabel("Number of clusters")
plt.title("Average silhouette plot over clusters", size=15)
plt.show()

- The average silhouette peaks at 4 clusters, in line with the elbow, so the final K-means solution uses 4.

#### **Using Sklearn:**

In [ ]:
# final clusters solution:

number_clusters = 4
kmclust = KMeans(n_clusters=number_clusters, init='k-means++', n_init=20, random_state=random_state)
allDistances = kmclust.fit_transform(X[num_vars])
km_labels = kmclust.fit_predict(X[num_vars])

- *init='k-means++'* initializes the centroids to be (generally) distant from each other, leading to probably better results than random initialization.  

- *n_init=n* allows to initialize KMeans n times and pick the best clustering in terms of Inertia. 

In [ ]:
# Characterizing the final clusters:
X_kmeans = pd.concat((X, pd.Series(km_labels, index=X.index, name='km_labels')), axis=1)

In [ ]:
# Characterizing the final clusters:
X_beforeNorm_kmeans = pd.concat((X_beforeNorm, pd.Series(km_labels, index=X_beforeNorm.index,
                                                         name='km_labels')), axis=1)
X_beforeNorm_kmeans.groupby('km_labels').mean().T

**Plotting clusters cardinality, magnitude, and cardinality vs magnitude:** 

In [ ]:
get_cluster_analysis(X_kmeans, "km_labels", allDistances)

**Visualizing the clusters sizes and distances in 2D:**

In [ ]:
visualizer = InterclusterDistance(kmclust)
visualizer.fit(X[num_vars])
visualizer.show()
plt.show()

#### **Using Scipy:**

In [ ]:
# computing K-Means with K clusters.

codebook, distortion = kmeans(X[num_vars], k_or_guess=number_clusters, iter=20)

- **codebook:** A k by N array of k centroids. The ith centroid codebook[i] is represented with the code i. The centroids and codes generated represent the lowest distortion seen, not necessarily the globally minimal distortion. Note that the number of centroids is not necessarily the same as the k_or_guess parameter, because centroids assigned to no observations are removed during iterations.

- **distortion:** The mean (non-squared) Euclidean distance between the observations passed and the centroids generated. Note the difference to the standard definition of distortion in the context of the k-means algorithm, which is the sum of the squared distances.

In [ ]:
# assign each sample to a cluster
code, dist = vq(X[num_vars], codebook)

- **code:** A length M array holding the code book index for each observation.

- **dist:** The distortion (distance) between the observation and its nearest code.

In [ ]:
# Characterizing the final clusters:
X_kmeans = pd.concat((X, pd.Series(code, index=X.index, name='km_labels')), axis=1)

In [ ]:
# Characterizing the final clusters:
X_beforeNorm_kmeans = pd.concat((X_beforeNorm, pd.Series(code, index=X_beforeNorm.index,
                                                         name='km_labels')), axis=1)
X_beforeNorm_kmeans.groupby('km_labels').mean().T

__Clusters Profiling:__

- Each column of the table above is one cluster's mean on the original scale.

**Clusters R²:**

In [ ]:
r2_kmm =  get_r2(X_kmeans, "km_labels")
print("Cluster solution with R² of %0.4f" % r2_kmm);

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_kmeans[num_vars + ['km_labels']], 'km_labels').drop('km_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

## __Visualization of KMeans clusters using Tsne:__

In [ ]:
Tsne_km = TSNE(perplexity=30, random_state=random_state).fit_transform(X_kmeans[num_vars])
pd.DataFrame(Tsne_km).plot.scatter(x=0, y=1, c = X_kmeans['km_labels'],
                                      colormap='viridis', figsize=(15,10))
plt.show();

- t-SNE projection of the clustering variables, coloured by cluster.

---

# **6. Hierarchical Clustering:**

### Different types of linkage

![Comparing different hierarchical linkage methods](https://scikit-learn.org/stable/_images/sphx_glr_plot_linkage_comparison_001.png)

### Characteristics:
- *bottom up approach*: each observation starts in its own cluster, and clusters are successively merged together.
- *greedy/local algorithm*: at each iteration tries to minimize the distance of cluster merging.
- *no realocation*: after an observation is assigned to a cluster, it can no longer change.
- *deterministic*: you always get the same answer when you run it.
- *scalability*: can be *very slow* for a large number of observations.

### __Defining the linkage method to choose:__

**We know that:**
$$SS_{t} = SS_{w} + SS_{b}$$

---

$$SS_{t} = \sum\limits_{i = 1}^n {{{({x_i} - \overline x )}^2}}$$

$$SS_{w} = \sum\limits_{k = 1}^K {\sum\limits_{i = 1}^{{n_k}} {{{({x_i} - {{\overline x }_k})}^2}} }$$

$$SS_{b} = \sum\limits_{k = 1}^K {{n_k}{{({{\overline x }_k} - \overline x )}^2}}$$

, where $n$ is the total number of observations, $x_i$ is the vector of the $i^{th}$ observation, $\overline x$ is the centroid of the data, $K$  is the number of clusters, $n_k$ is the number of observations in the $k^{th}$ cluster and $\overline x_k$ is the centroid of the $k^{th}$ cluster.

In [ ]:
def get_r2_hc(df, link_method, max_nclus, min_nclus=1, dist="euclidean"):
    """This function computes the R2 for a set of cluster solutions given by the application of a hierarchical
    method.
    The R2 is a measure of the homogenity of a cluster solution. It is based on SSt = SSw + SSb and R2 = SSb/SSt or
    R2 = 1 - SSw/SSt. 
    
    Parameters:
    df (DataFrame): Dataset to apply clustering.
    link_method (str): either "ward", "complete", "average", "single"
    max_nclus (int): maximum number of clusters to compare the methods.
    min_nclus (int): minimum number of clusters to compare the methods. Defaults value is 1.
    dist (str): distance to use to compute the clustering solution. Must be a valid distance. 
    Defaults to "euclidean".
    
    Returns:
    ndarray: R2 values for the range of cluster solutions
    """    
    def get_ss(df):
        """Computes the sum of squares for all variables given a dataset
        """
        ss = np.sum(df.var()*(df.count() - 1))
        return ss  # return sum of sum of squares of each df variable.
    
    sst = get_ss(df)  # get total sum of squares.
    
    r2 = []  # To store the R2 metrics for each cluster solution.
    
    for i in range(min_nclus, max_nclus+1):  # iterate over desired ncluster range
        
        cluster = AgglomerativeClustering(n_clusters=i, affinity=dist, linkage=link_method)
        hclabels = cluster.fit_predict(df) #get cluster labels
        df_concat = pd.concat((df, pd.Series(hclabels, index = df.index,
                                             name='labels')), axis=1)  # concat df with labels
        ssw_labels = df_concat.groupby(by='labels').apply(get_ss)  # compute ssw for each cluster labels
        ssb = sst - np.sum(ssw_labels)  # remember: SST = SSW + SSB
        r2.append(ssb/sst)  # save the R2 of the given cluster solution
        
    return np.array(r2)

In [ ]:
# Preparing the inputs.
hc_methods = ["ward", "complete", "average", "single"]
# Calling the function defined above to obtain the R2 statistic for each hc_method
max_nclus = 10
r2_hc_methods = np.vstack([get_r2_hc(df=X[num_vars], link_method=link, max_nclus=max_nclus) for link in hc_methods]).T
r2_hc_methods = pd.DataFrame(r2_hc_methods, index=range(1, max_nclus + 1), columns=hc_methods)

sns.set()
# Plotting the data:
fig = plt.figure(figsize=(10,7))
sns.lineplot(data=r2_hc_methods, linewidth=2.5, markers=["o"]*4)

# Finalize the plot
fig.suptitle("R2 plot for various hierarchical methods", fontsize=21)
plt.gca().invert_xaxis()  # invert x axis.
plt.legend(title="HC methods", title_fontsize=11)
plt.xticks(range(1, max_nclus + 1))
plt.xlabel("Number of clusters", fontsize=13)
plt.ylabel("R2 metric", fontsize=13)

plt.show()

- We can see that _ward linkage_ provides the best results.

**Ward linkage**: minimizes the sum of squared differences within all clusters. It is a variance-minimizing approach and in this sense is similar to the k-means objective function but tackled with an agglomerative hierarchical approach.

### __Defining the number of clusters:__
Where is the **first big jump** on the Dendrogram?

Inspired from: https://joernhees.de/blog/2015/08/26/scipy-hierarchical-clustering-and-dendrogram-tutorial/

In [ ]:
linked = linkage(X[num_vars], method="ward", metric="euclidean")

# calculate full dendrogram
plt.figure(figsize=(25, 10))
plt.title('Hierarchical Clustering Dendrogram', fontsize=21)
plt.xlabel('sample index')
plt.ylabel('distance', fontsize=13)
dendrogram(
    linked,
    leaf_rotation=90.,  # rotates the x axis labels
    leaf_font_size=8.,  # font size for the x axis labels
)
plt.show()

In [ ]:
def fancy_dendrogram(*args, **kwargs):
    max_d = kwargs.pop('max_d', None)
    if max_d and 'color_threshold' not in kwargs:
        kwargs['color_threshold'] = max_d
    annotate_above = kwargs.pop('annotate_above', 0)

    ddata = dendrogram(*args, **kwargs)

    if not kwargs.get('no_plot', False):
        plt.title('Hierarchical Clustering Dendrogram (truncated)')
        plt.xlabel('sample index or (cluster size)')
        plt.ylabel('distance')
        for i, d, c in zip(ddata['icoord'], ddata['dcoord'], ddata['color_list']):
            x = 0.5 * sum(i[1:3])
            y = d[1]
            if y > annotate_above:
                plt.plot(x, y, 'o', c=c)
                plt.annotate("%.3g" % y, (x, y), xytext=(0, -5),
                             textcoords='offset points',
                             va='top', ha='center')
        if max_d:
            plt.axhline(y=max_d, c='k')
    return ddata

In [ ]:
max_d = 90

fancy_dendrogram(
    linked,
    truncate_mode='lastp',
    p=12,
    leaf_rotation=90.,
    leaf_font_size=12.,
    show_contracted=True,
    annotate_above=10,
    max_d=max_d,  # plot a horizontal cut-off line
)
plt.show()

In [ ]:
# Getting the silhouette score.

avg_silhouette_hc=get_avg_silhouette(X[num_vars], AgglomerativeClustering(linkage="ward", 
                                                                          affinity="euclidean"))

In [ ]:
# The average silhouette plot
plt.figure(figsize=(9,5))
plt.plot(range(2,11), avg_silhouette_hc, marker=".", markersize=12)
plt.ylabel("Average silhouette")
plt.xlabel("Number of clusters")
plt.title("Average silhouette plot over clusters", size=15)
plt.show()

The average silhouette peaks at 4 clusters, and the dendrogram's largest jump also suggests 4.

### __Final Hierarchical clustering solution:__

#### __Using sklearn:__

In [ ]:
# 4 cluster solution:
link = 'ward'
distance = 'euclidean'
n_clusters=4
hcclust = AgglomerativeClustering(linkage=link, affinity=distance, 
                                  n_clusters=n_clusters, compute_distances=True)
hc_labels = hcclust.fit_predict(X[num_vars])

In [ ]:
# Characterizing the clusters
X_hc = pd.concat((X, pd.Series(hc_labels, index=X.index, name='hc_labels')), axis=1)

**We verify the relation bellow for each variable in our dataframe:**
$$SS_{t} = SS_{w} + SS_{b}$$

In [ ]:
# Computing SST
X_tmp = X_hc[num_vars+["hc_labels"]].values
sst = np.sum(np.square(X_tmp - X_tmp.mean(axis=0)), axis=0)

# Computing SSW
ssw_iter = []
for i in np.unique(hc_labels):
    X_k = X_tmp[hc_labels == i]
    ssw_iter.append(np.sum(np.square(X_k - X_k.mean(axis=0)), axis=0))
ssw = np.sum(ssw_iter, axis=0)

# Computing SSB
ssb_iter = []
for i in np.unique(hc_labels):
    X_k = X_tmp[hc_labels == i]
    ssb_iter.append(X_k.shape[0] * np.square(X_k.mean(axis=0) - X_tmp.mean(axis=0)))
ssb = np.sum(ssb_iter, axis=0)

# Verifying the formula
np.round(sst) == np.round((ssw + ssb))

In [ ]:
# Characterizing the clusters.
X_beforeNorm_hc = pd.concat((X_beforeNorm, pd.Series(hc_labels, index=X_beforeNorm.index, name='hc_labels')), axis=1)
X_beforeNorm_hc.groupby('hc_labels').mean().T

#### __Using Scipy:__

In [ ]:
k=4
hc_labels = fcluster(linked, k, criterion='maxclust')

In [ ]:
# Characterizing the clusters
X_hc = pd.concat((X, pd.Series(hc_labels, index=X.index, name='hc_labels')), axis=1)

In [ ]:
# Characterizing the clusters.
X_beforeNorm_hc = pd.concat((X_beforeNorm, pd.Series(hc_labels, index=X_beforeNorm.index, name='hc_labels')), axis=1)
X_beforeNorm_hc.groupby('hc_labels').mean().T

__Clusters Profiling:__

- Each column of the table above is one cluster's mean on the original scale.

In [ ]:
plt.figure(figsize=(10, 7))

# We need to define the centroids of our clusters.
centroids = X_hc.groupby('hc_labels', as_index=False).mean()
pd.plotting.parallel_coordinates(centroids, class_column = "hc_labels", colormap="viridis",lw=2)

plt.xticks(fontsize=10, rotation=90)
plt.show()

**Clusters R²:**

In [ ]:
r2_hc =  get_r2(X_hc, "hc_labels")
print("Cluster solution with R^2 of %0.4f" % r2_hc);

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_hc[num_vars + ['hc_labels']], 'hc_labels').drop('hc_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

## __Visualization of Hierarchical clusters using TSNE:__

In [ ]:
Tsne_hc = TSNE(perplexity=30, random_state=random_state).fit_transform(X_hc[num_vars])

pd.DataFrame(Tsne_hc).plot.scatter(x=0, y=1, c = X_hc['hc_labels'],
                                      colormap='viridis', figsize=(15,10))
plt.show();

- t-SNE projection of the clustering variables, coloured by cluster.

___

# **7. Density Based Clustering:**


## **7.1 Mean Shift Clustering:**

### _Characteristics:_
- No need to define number of clusters apriori.
- Can detect clusters of any shape.
- Robust to outliers.
- Depends on the bandwidth hyperparameter.
- **Main drawback**: Poor scalability (on both the algorithm and in estimating the bandwidth).

In [ ]:
# The following bandwidth can be automatically detected using (we need to set a quantile).
# Based on distance to nearest neighbors for all observations.

bandwidth = estimate_bandwidth(X[num_vars], quantile=0.06, n_samples=50, 
                               random_state=random_state, n_jobs=-1)
bandwidth

In [ ]:
# Perform mean-shift clustering with bandwidth set using estimate_bandwidth
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True, n_jobs=-1)
ms_labels = ms.fit_predict(X[num_vars])

ms_n_clusters = len(np.unique(ms_labels))
print("Number of estimated clusters : %d" % ms_n_clusters)

In [ ]:
# Concatenating the labels to X
X_ms = pd.concat([X, pd.Series(ms_labels, index=X.index, name="ms_labels")], axis=1)

In [ ]:
# Concatenating the labels to X
X_beforeNorm_ms = pd.concat([X_beforeNorm, pd.Series(ms_labels, index=X_beforeNorm.index, name="ms_labels")], axis=1)
X_beforeNorm_ms.groupby('ms_labels').mean().T

__Clusters Profilling:__

- Each column of the table above is one cluster's mean on the original scale.

In [ ]:
plt.figure(figsize=(10, 7))

# We need to define the centroids of our clusters.
centroids = X_ms.groupby('ms_labels', as_index=False).mean()
pd.plotting.parallel_coordinates(centroids, class_column = "ms_labels", colormap="viridis",lw=2)

plt.xticks(fontsize=10, rotation=90)
plt.show();

**Clusters R²:**

In [ ]:
r2_ms =  get_r2(X_ms[num_vars + ["ms_labels"]], "ms_labels")
print("Cluster solution with R^2 of %0.4f" % r2_ms);

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_ms[num_vars + ['ms_labels']], 'ms_labels').drop('ms_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

### Visualization of Mean shift clusters using TSNE:

In [ ]:
Tsne_ms = TSNE(perplexity=30, random_state=random_state).fit_transform(X_ms[num_vars])

pd.DataFrame(Tsne_ms).plot.scatter(x=0, y=1, c = X_ms['ms_labels'],
                                      colormap='viridis', figsize=(15,10))
plt.show();

- t-SNE projection of the clustering variables, coloured by cluster.

___

## **7.2 DBSCAN (Density-based spatial clustering of applications with noise):**

### Characteristics:
- No need to define number of clusters apriori.
- Resistant to noise and outliers.
- Can identify outliers.
- Can handle clusters of different shapes and sizes.
- Depends highly on the epsilon hyperparameter and it can be hard to tune.
- Does not work well with clusters of varying densities.

### Defining eps and min_samples:
- **MinPts**: As a rule of thumb, **minPts = 2 x dim** can be used, but it may be necessary to choose larger values for very large data, for noisy data or for data that contains many duplicates.

- **ε**: The value for ε can then be chosen by using a **k-distance graph**, plotting the distance to the kth (k = minPts - 1) nearest neighbor ordered from the largest to the smallest value. Good values of ε are where this plot shows an **"elbow"**: if ε is chosen much too small, a large part of the data will not be clustered; whereas for a too high value of ε, clusters will merge and the majority of objects will be in the same cluster. **The assumption is that for points in a cluster, their k nearest neighbors are at roughly the same distance**. Noise points have their k-th nearest neighbors at farther distance

In [ ]:
# Perform DBSCAN clustering.

dbscan = DBSCAN(eps=2.3, min_samples=15, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X[num_vars])
Counter(dbscan_labels)

In [ ]:
dbscan_n_clusters = len(np.unique(dbscan_labels)) - int(-1 in dbscan_labels)
print("Number of estimated clusters : %d" % dbscan_n_clusters)

In [ ]:
# K-distance graph to find out the right eps value
neigh = NearestNeighbors(n_neighbors=20)
neigh.fit(X[num_vars])
distances, _ = neigh.kneighbors(X[num_vars])
distances = np.sort(distances[:, -1])
plt.plot(distances)
plt.show()

In [ ]:
# Concatenating the labels to X
X_dbscan = pd.concat([X, pd.Series(dbscan_labels, index=X.index, name="dbscan_labels")], axis=1)

In [ ]:
# Concatenating the labels to X
X_beforeNorm_dbscan = pd.concat([X_beforeNorm, pd.Series(dbscan_labels, index=X_beforeNorm.index, name="dbscan_labels")], axis=1)
X_beforeNorm_dbscan.groupby('dbscan_labels').mean().T

In [ ]:
# Detecting noise (potential outliers)
X_dbscan.loc[X_dbscan['dbscan_labels'] == -1].head(2)

__Clusters R²:__

In [ ]:
r2_dbscan =  get_r2(X_dbscan, "dbscan_labels")
print("Cluster solution with R^2 of %0.4f" % r2_dbscan)

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_dbscan[num_vars + ['dbscan_labels']], 'dbscan_labels').drop('dbscan_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

- The DBSCAN clustering algorithm gave us just one cluster.
- Some observations are considered as noise.

___

## **7.3 GMM (Gaussian Mixture Model):**
 
$$\mathcal{p(\vec{x})} \ = \ \sum_{i=1}^K \phi_i \mathcal{N}(\vec{x}|\vec{\mu_i}, \Sigma_i) \tag{eq1}$$
$$\mathcal{N}(\vec{x}|\vec{\mu_i}, \Sigma_i) \ = \ \frac{1}{\sqrt{{(2\pi)}^{K}|\Sigma_i|}}e^{-\frac{1}{2} (\vec{x} - \vec{\mu_i})^T \Sigma_i^{-1} (\vec{x} - \vec{\mu_i})} \tag{eq2}$$
$$\sum_{i=1}^K \phi_i \ = \ 1 \tag{eq3}$$

, where:
- $\phi_i$ is the component weight (scalar) for Component $i$ (probability of an observation being generated by Component $i$)
- $\vec{\mu_i}$ is the mean vector for Component $i$,
- $\Sigma_i$ is the Covariance matrix for Component $i$

---

- **(eq1)** gives the probability of a point $x$ given the estimated Gaussian mixture
- **(eq2)** is the probability density function of a multivariate Gaussian with mean $\vec{\mu_i}$ and covariance $\Sigma_i$
- **(eq3)** states that the sum of the component weights is 1, such that the total probability distribution normalizes to 1

### Characteristics:
- Assumes the data is generated from a mixture of finite number of Gaussian distributions with unknown parameters.
- Uses the EM (Expectation Maximization algorithm) to estimate the parameters.
- Provides a probability of each observation belonging to each cluster.
- Advantages over K-Means:
    - Can deal with spherical and elipsoid cluster shapes.
    - Number of components needs to be defined apriori.

In [ ]:
# Performing GMM clustering:
n_components=4
gmm = GaussianMixture(n_components=n_components, covariance_type='tied', n_init=20,
                      init_params='kmeans', random_state=random_state)
gmm_labels = gmm.fit_predict(X[num_vars])
labels_proba = gmm.predict_proba(X[num_vars])

**Let's look at the estimated parameters:**

In [ ]:
# The estimated component weights.
gmm.weights_

In [ ]:
# The estimated mean vectors of the Components.
gmm.means_

In [ ]:
# The estimated covariance matrices of the Components.
gmm.covariances_

### __Defining covariance_type:__
This hyperparameter controls the **degrees of freedom** in the shape of each cluster The more degrees of freedom we have the more complex shapes the model can fit and the more computationally expensive the model will be.

- `covariance_type="tied"` makes all components share the same general covariance matrix.

### __Defining the number of clusters:__

- **AIC**: estimates the relative amount of information lost by a model used to represent the data-generation process. The smaller the better.

- **BIC**: similar to AIC but penalizes more complex models (i.e. favors simpler models). The smaller the better.

In [ ]:
# Selecting number of components based on AIC and BIC
n_components = np.arange(1, 16)
models = [GaussianMixture(n_components=n, covariance_type='tied', 
                          n_init=20, random_state=random_state).fit(X[num_vars])
          for n in n_components]

bic_values = [m.bic(X[num_vars]) for m in models]
aic_values = [m.aic(X[num_vars]) for m in models]
plt.plot(n_components, bic_values, label='BIC')
plt.plot(n_components, aic_values, label='AIC')
plt.legend(loc='best')
plt.xlabel('n_components')
plt.xticks(n_components)
plt.show()

**Note**: the AIC and BIC measures can also be used to select diferent hyperparameters such as the covariance_type

- AIC and BIC keep falling up to about 14 components, so they give no clear choice here; the silhouette below is used instead.

**Silhouette Coefficient formula for a single sample:**
$$s = \frac{b - a}{max(a, b)}$$
, where:
- $a$: The mean distance between a sample and all other points in the same cluster.
- $b$: The mean distance between a sample and all other points in the next nearest cluster

In [ ]:
# Adapted from:
# https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.htmlsphx-glr-auto-examples-cluster-plot-kmeans-silhouette-analysis-py

# Storing average silhouette metric
avg_silhouette = []
for nclus in range_clusters:
    # Skip nclus == 1
    if nclus == 1:
        continue
    
    # Create a figure
    fig = plt.figure(figsize=(13, 7))

    # Initialize the gmm object with n_clusters value and a random generator
    # seed of 10 for reproducibility.
    gmmclust = GaussianMixture(n_components=nclus, covariance_type='full', n_init=20, 
                      init_params='kmeans', random_state=random_state)
    cluster_labels = gmmclust.fit_predict(X[num_vars])

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed clusters
    silhouette_avg = silhouette_score(X[num_vars], cluster_labels)
    avg_silhouette.append(silhouette_avg)
    print(f"For n_clusters = {nclus}, the average silhouette_score is : {silhouette_avg}")

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X[num_vars], cluster_labels)

    y_lower = 10
    for i in range(nclus):
        # Aggregate the silhouette scores for samples belonging to cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()
        
        # Get y_upper to demarcate silhouette y range size
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        # Filling the silhouette
        color = cm.nipy_spectral(float(i) / nclus)
        plt.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)

        # Label the silhouette plots with their cluster numbers at the middle
        plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    plt.title("The silhouette plot for the various clusters.")
    plt.xlabel("The silhouette coefficient values")
    plt.ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    plt.axvline(x=silhouette_avg, color="red", linestyle="--")
    
    # The silhouette coefficient can range from -1, 1
    xmin, xmax = np.round(sample_silhouette_values.min() -0.1, 2), np.round(sample_silhouette_values.max() + 0.1, 2)
    plt.xlim([xmin, xmax])
    
    # The (nclus+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    plt.ylim([0, len(X[num_vars]) + (nclus + 1) * 10])

    plt.yticks([])  # Clear the yaxis labels / ticks
    plt.xticks(np.arange(xmin, xmax, 0.1))

In [ ]:
# The average silhouette plot
plt.figure(figsize=(9,5))
plt.plot(range(2,11), avg_silhouette, marker=".", markersize=12)
plt.ylabel("Average silhouette")
plt.xlabel("Number of clusters")
plt.title("Average silhouette plot over clusters", size=15)
plt.show()

The average silhouette peaks at 4 components, so the final model uses 4.

#### __Final Gaussian Mixture Model clustering solution:__

In [ ]:
# Performing GMM clustering
n_components=4
gmm = GaussianMixture(n_components=n_components, covariance_type='full', n_init=20, 
                      init_params='kmeans', random_state=random_state)
gmm_labels = gmm.fit_predict(X[num_vars])

In [ ]:
# Concatenating the labels to X
X_gmm = pd.concat([X, pd.Series(gmm_labels, index=X.index, name="gmm_labels")], axis=1)

In [ ]:
# Concatenating the labels to X
X_beforeNorm_gmm = pd.concat([X_beforeNorm, pd.Series(gmm_labels, 
                              index=X_beforeNorm.index, name="gmm_labels")], axis=1)
X_beforeNorm_gmm.groupby('gmm_labels').mean().T

__Clusters Profiling:__

- Each column of the table above is one cluster's mean on the original scale.

In [ ]:
plt.figure(figsize=(10, 7))

# We need to define the centroids of our clusters.
centroids = X_gmm.groupby('gmm_labels', as_index=False).mean()
pd.plotting.parallel_coordinates(centroids, class_column = "gmm_labels", colormap="viridis",lw=2)

plt.xticks(fontsize=10, rotation=90)
plt.show()

**Clusters R²:**

In [ ]:
r2_gmm =  get_r2(X_gmm, "gmm_labels")
print("Cluster solution with R^2 of %0.4f" % r2_gmm);

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_gmm[num_vars + ['gmm_labels']], 'gmm_labels').drop('gmm_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

### Visualization of Gaussian Mixture Model clusters using TSNE:

In [ ]:
Tsne_gmm = TSNE(perplexity=30, random_state=random_state).fit_transform(X_gmm[num_vars])
pd.DataFrame(Tsne_gmm).plot.scatter(x=0, y=1, c = X_gmm['gmm_labels'],
                                      colormap='viridis', figsize=(15,10))
plt.show();

- t-SNE projection of the clustering variables, coloured by cluster.

___

# **8. Clustering by perspectives:**
We split the variables into perspectives.

In [ ]:
# Split variables into perspectives (requires critical thinking and domain knowledge)
demographic_features = [
    'MonthSal',
    'Age',
    'YearsAsCustomer', 
    'CustMonVal',
    'ClaimsRate',  
]

Prem_features = [
    'PremMotor',
    'PremHousehold',
    'PremHealth',
    'PremLife',
    'PremWork', 
]

X_perspective = X.copy(deep=True)
X_dem = X_perspective[demographic_features].copy(deep=True)
X_prm = X_perspective[Prem_features].copy(deep=True)

## **Testing K-means and Hierarchical clustering:**

In [ ]:
def get_r2_scores(df, clusterer, min_k=2, max_k=10):
    """
    Loop over different values of k. To be used with sklearn clusterers.
    """
    r2_clust = {}
    for n in range(min_k, max_k):
        clust = clone(clusterer).set_params(n_clusters=n)
        labels = clust.fit_predict(df)
        r2_clust[n] = get_r2(df, labels)
    return r2_clust

In [ ]:
# Set up the clusterers
kmeans = KMeans(
    init='k-means++',
    n_init=20,
    random_state=random_state
)

hierarchical = AgglomerativeClustering(
    affinity='euclidean'
)

### **Finding the optimal clusters on demographic variables:**

In [ ]:
# Obtaining the R² scores for each cluster solution on demographic variables
r2_scores = {}
r2_scores['kmeans'] = get_r2_scores(X_dem, kmeans)

for link in ['complete', 'average', 'single', 'ward']:
    r2_scores[link] = get_r2_scores(
        X_dem, hierarchical.set_params(linkage=link)
    )

pd.DataFrame(r2_scores)

In [ ]:
# Visualizing the R² scores for each cluster solution on demographic variables
pd.DataFrame(r2_scores).plot.line(figsize=(10,7))

plt.title("Demographic Variables:\nR² plot for various clustering methods\n", fontsize=21)
plt.legend(title="Cluster methods", title_fontsize=11)
plt.xlabel("Number of clusters", fontsize=13)
plt.gca().invert_xaxis()  # invert x axis
plt.ylabel("R² metric", fontsize=13)
plt.show();

- CHECK_PLACEHOLDER

### **Repeat the process for premium variables:**

In [ ]:
# Obtaining the R² scores for each cluster solution on product variables
r2_scores = {}
r2_scores['kmeans'] = get_r2_scores(X_prm, kmeans)

for link in ['complete', 'average', 'single', 'ward']:
    r2_scores[link] = get_r2_scores(
        X_prm, hierarchical.set_params(linkage=link)
    )

pd.DataFrame(r2_scores)

In [ ]:
# Visualizing the R² scores for each cluster solution on product variables
pd.DataFrame(r2_scores).plot.line(figsize=(10,7))

plt.title("Product Variables:\nR2 plot for various clustering methods\n", fontsize=21)
plt.legend(title="Cluster methods", title_fontsize=11)
plt.xlabel("Number of clusters", fontsize=13)
plt.gca().invert_xaxis()  # invert x axis
plt.ylabel("R2 metric", fontsize=13)
plt.show()

- CHECK_PLACEHOLDER

## **Merging the Perspectives:**
- How can we merge different cluster solutions?

In [ ]:
# Applying the right clustering (algorithm and number of clusters) for each perspective

kmeans_dem = KMeans(
    n_clusters=4,
    init='k-means++',
    n_init=20,
    random_state=random_state
)
dem_labels = kmeans_dem.fit_predict(X_dem)


kmeans_prm = KMeans(
    n_clusters=3,
    init='k-means++',
    n_init=20,
    random_state=random_state
)

prm_labels = kmeans_prm.fit_predict(X_prm)

X_dem['dem_labels'] = dem_labels
X_prm['prm_labels'] = prm_labels

X_perspective["dem_labels"] = dem_labels
X_perspective["prm_labels"] = prm_labels

In [ ]:
# Counting label frequencies (contigency table)
X_perspective.groupby(['prm_labels', 'dem_labels'])\
    .size()\
    .to_frame()\
    .reset_index()\
    .pivot('dem_labels', 'prm_labels', 0)

### **Merging the clusters using Hierarchical clustering:**

In [ ]:
# Centroids of the concatenated cluster labels
X_perspective_centroids = X_perspective.groupby(['dem_labels', 'prm_labels'])\
    [num_vars].mean()
X_perspective_centroids

In [ ]:
# Using Hierarchical clustering to merge the concatenated cluster centroids
hclust = AgglomerativeClustering(
    linkage='ward', 
    affinity='euclidean', 
    distance_threshold=0, 
    n_clusters=None
)
hclust_labels = hclust.fit_predict(X_perspective_centroids)

In [ ]:
linked = linkage(X_perspective_centroids, method="ward", metric="euclidean")

# calculate full dendrogram
plt.figure(figsize=(25, 10))
plt.title('Hierarchical Clustering Dendrogram', fontsize=21)
plt.xlabel('sample index')
plt.ylabel('distance', fontsize=13)
dendrogram(
    linked,
    leaf_rotation=90.,  # rotates the x axis labels
    leaf_font_size=8.,  # font size for the x axis labels
)
plt.show()

In [ ]:
max_d = 5

fancy_dendrogram(
    linked,
    truncate_mode='lastp',
    p=12,
    leaf_rotation=90.,
    leaf_font_size=12.,
    show_contracted=True,
    annotate_above=10,
    max_d=max_d,  # plot a horizontal cut-off line
)
plt.show()

In [ ]:
# Re-running the Hierarchical clustering based on the correct number of clusters

nclust=3

hclust = AgglomerativeClustering(
    linkage='ward', 
    affinity='euclidean', 
    n_clusters=nclust
)

hclust_labels = hclust.fit_predict(X_perspective_centroids)
X_perspective_centroids['hclust_labels'] = hclust_labels

X_perspective_centroids  # centroid's cluster labels

In [ ]:
X_ = X_perspective.copy()

# Mapper between concatenated clusters and hierarchical clusters
cluster_mapper = X_perspective_centroids['hclust_labels'].to_dict()

# Mapping the hierarchical clusters on the centroids to the observations
X_['merged_labels'] = X_.apply(
    lambda row: cluster_mapper[
        (row['dem_labels'], row['prm_labels'])
    ], axis=1
)

In [ ]:
X_beforeNorm_merged = X_beforeNorm.copy(deep=True)
X_beforeNorm_merged['merged_labels'] = X_['merged_labels']

In [ ]:
# Merged cluster centroids:

X_beforeNorm_merged.groupby('merged_labels').mean().T

__Clusters Profiling:__

- Each column of the table above is one cluster's mean on the original scale.

## **Cluster Analysis:**

In [ ]:
def cluster_profiles(df, label_columns, figsize, compar_titles=None):
    """
    Pass df with labels columns of one or multiple clustering labels. 
    Then specify this label columns to perform the cluster profile according to them.
    """
    if compar_titles == None:
        compar_titles = [""]*len(label_columns)
        
    sns.set()
    fig, axes = plt.subplots(nrows=len(label_columns), ncols=2, figsize=figsize, squeeze=False)
    for ax, label, titl in zip(axes, label_columns, compar_titles):
        # Filtering df
        drop_cols = [i for i in label_columns if i!=label]
        dfax = df.drop(drop_cols, axis=1)
        
        # Getting the cluster centroids and counts
        centroids = dfax.groupby(by=label, as_index=False).mean()
        counts = dfax.groupby(by=label, as_index=False).count().iloc[:,[0,1]]
        counts.columns = [label, "counts"]
        
        # Setting Data
        pd.plotting.parallel_coordinates(centroids, label, color=sns.color_palette(), ax=ax[0])
        sns.barplot(x=label, y="counts", data=counts, ax=ax[1])

        #Setting Layout
        handles, _ = ax[0].get_legend_handles_labels()
        cluster_labels = ["Cluster {}".format(i) for i in range(len(handles))]
        ax[0].annotate(text=titl, xy=(0.95,1.1), xycoords='axes fraction',
                        fontsize=13, fontweight = 'heavy') 
        ax[0].legend(handles, cluster_labels) # Adaptable to number of clusters
        ax[0].axhline(color="black", linestyle="--")
        ax[0].set_title("Cluster Means - {} Clusters".format(len(handles)), fontsize=13)
        ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=-20)
        ax[1].set_xticklabels(cluster_labels)
        ax[1].set_xlabel("")
        ax[1].set_ylabel("Absolute Frequency")
        ax[1].set_title("Cluster Sizes - {} Clusters".format(len(handles)), fontsize=13)
    
    plt.subplots_adjust(hspace=0.4, top=0.90)
    plt.suptitle("Cluster Simple Profilling", fontsize=23)
    plt.show()

In [ ]:
# Profilling each cluster (product, behavior, merged)
cluster_profiles(
    df = X_[num_vars + ['prm_labels', 'dem_labels', 'merged_labels']], 
    label_columns = ['prm_labels', 'dem_labels', 'merged_labels'], 
    figsize = (28, 13), 
    compar_titles = ["prm_labels", "dem_labels", "merged_labels"]
)

**Clusters R²:**

What proportion of each variables total SS is explained between clusters?

In [ ]:
r2_merged =  get_r2(X_, "merged_labels")
print("Cluster solution with R^2 of %0.4f" % r2_merged);

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_[num_vars + ['merged_labels']], 'merged_labels').drop('merged_labels').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

## __Clusters visualization using t-SNE:__

In [ ]:
# This is step can be quite time consuming
two_dim = TSNE(random_state=random_state).fit_transform(X_[num_vars])

# t-SNE visualization
pd.DataFrame(two_dim).plot.scatter(x=0, y=1, c=X_['merged_labels'], colormap='viridis', figsize=(15,10))
plt.show()

- t-SNE projection of the clustering variables, coloured by cluster.

## **Assess feature importance:**


### **Using a Decision Tree:**
We get the normalized total reduction of the criterion (gini or entropy) brought by that feature (also known as Gini importance).

In [ ]:
# Preparing the data
X_tree = X_.drop(columns=['prm_labels','dem_labels','merged_labels'])
y = X_["merged_labels"]

# Splitting the data:
X_train, X_test, y_train, y_test = train_test_split(
    X_tree, y, test_size=0.2, random_state=random_state
)

# Fitting the decision tree
dt = DecisionTreeClassifier(random_state=random_state, max_depth=5)
dt.fit(X_train, y_train)
print("It is estimated that in average, we are able to predict {0:.2f}% of the customers correctly".format(dt.score(X_test, y_test)*100))

In [ ]:
# Assessing feature importance
pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(axis=0, ascending=False)

- CHECK_PLACEHOLDER

In [ ]:
# Visualizing the decision tree

dot_data = export_graphviz(dt, out_file=None, 
                           feature_names=X_tree.columns.to_list(),
                           filled=True,
                           rounded=True,
                           special_characters=True)  

graphviz.Source(dot_data)

# **9. Self Organizing Map:**

A SOM is an artifcial neural network composed by a grid of output neurons
connected to an input layer.

This type of neural network uses an unsupervised learning algorithm to
fnd clusters in data without any privileged knowledge a priori.

The algorithm maps a multidimensional training set in a 2D grid of neurons
in a way that preserves the original topological relationships.

Inspired by: https://indico.lip.pt/event/396/contributions/898/attachments/995/1133/Franco_BigData.pdf

In [ ]:
# This som implementation does not have a random seed parameter
# We're going to set it up ourselves
np.random.seed(random_state)

# Notice that the SOM did not converge - We're under a time constraint for this class
sm = sompy.SOMFactory().build(
    X[num_vars].values ,  
    mapsize=(50, 50), 
    initialization='random',
    neighborhood='gaussian',
    training='batch',
    lattice='hexa',
    component_names=num_vars
)
sm.train(n_job=4, verbose='info', train_rough_len=100, train_finetune_len=100)

### __K-Means on top of SOM units:__

- We can visualize the clusters formed by labeled neurons:

In [ ]:
# Perform K-Means clustering on top of the 2500 units (sm.codebook.matrix output)
nclus=3
kmeans = KMeans(n_clusters=nclus, init='k-means++', n_init=20, random_state=random_state)
nodeclus_labels_km = kmeans.fit_predict(sm.codebook.matrix)
sm.cluster_labels = nodeclus_labels_km

hits  = HitMapView(12, 12, "hitmap", text_size=10) 
hits.show(sm, anotate=True, onlyzeros=False, labelsize=7, cmap="Pastel1")

plt.show()

- Hit map of the SOM units, coloured by the cluster of each unit.

### __Hierarchical Clustering on top of SOM units:__

- We can visualize the clusters formed by labeled neurons:

In [ ]:
# Perform Hierarchical clustering on top of the 2500 units (sm.codebook.matrix output)
nclus=3
hierclust = AgglomerativeClustering(n_clusters=nclus, linkage='ward')
nodeclus_labels_hc = hierclust.fit_predict(sm.codebook.matrix)
sm.cluster_labels = nodeclus_labels_hc

hits = HitMapView(12, 12, "hitmap", text_size=10) 
hits.show(sm, anotate=True, onlyzeros=False, labelsize=7, cmap="Pastel1")

plt.show()

- Hit map of the SOM units, coloured by the cluster of each unit.

### __Final SOM Clustering solution:__

In [ ]:
# Check the nodes and and respective clusters
nodes = sm.codebook.matrix

df_nodes = pd.DataFrame(nodes, columns=X[num_vars].columns)
df_nodes['label'] = nodeclus_labels_km
df_nodes.head(2)

In [ ]:
# Obtaining SOM's BMUs (Best Matching units) labels
df_bmus = pd.DataFrame(
    np.concatenate((X, np.expand_dims(sm._bmu[0],1)), axis=1),
    index=X.index, columns=np.append(X.columns,"BMU")
)
df_bmus.tail()

In [ ]:
# Get cluster labels for each observation
X_som = df_bmus.merge(df_nodes['label'], 'left', left_on="BMU", right_index=True)
X_som.head(2)

In [ ]:
X_beforeNorm_som = X_beforeNorm.copy(deep=True)
X_beforeNorm_som[["BMU", "label"]] = X_som[["BMU", "label"]]

In [ ]:
# Characterizing the final clusters
X_beforeNorm_som.drop(columns='BMU').groupby('label').mean().T

__Clusters Profiling:__

- Each column of the table above is one cluster's mean on the original scale.

In [ ]:
plt.figure(figsize=(10, 7))

# We need to define the centroids of our clusters.
centroids = X_som.drop(columns='BMU').groupby('label', as_index=False).mean()
pd.plotting.parallel_coordinates(centroids, class_column = "label", colormap="viridis",lw=2)

plt.xticks(fontsize=10, rotation=90)
plt.show()

**Clusters R²:**

In [ ]:
r2_som =  get_r2(X_som, "label")
print("Cluster solution with R^2 of %0.4f" % r2_som, "\n");

# We are essentially decomposing the R² into the R² for each variable:

get_r2_variables(X_som[num_vars + ['label']], 'label').drop('label').\
                 sort_values(axis=0, ascending=False)

- R2_PLACEHOLDER

## __Clusters visualization using t-SNE:__

In [ ]:
# This is step can be quite time consuming
two_dim = TSNE(random_state=random_state).fit_transform(X_som[num_vars])

# t-SNE visualization
pd.DataFrame(two_dim).plot.scatter(x=0, y=1, c=X_som['label'], colormap='viridis', figsize=(15,10))
plt.show()

- t-SNE projection of the clustering variables, coloured by cluster.

In [ ]:
t2 = time.perf_counter()#
print('time taken to run:',(t2-t1)/60.0)

___